# Memory-Efficient PPO / RLHF with a Single LoRA Model

This companion notebook runs the repository's example and memory benchmark in Google Colab. The technique uses one frozen base model with adapters off for reference scoring and on for the PPO actor. It is already used in Hugging Face TRL; see the repository README for prior work and limitations.

Select a CUDA GPU runtime before running the benchmark. The 3B model requires several GB of disk and GPU memory.

## 1. Install dependencies and clone the repository

In [ ]:
!pip install -q transformers peft accelerate
from pathlib import Path
import subprocess
repo = Path('/content/ppo-vram-halver')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Bender1011001/ppo-vram-halver.git', str(repo)], check=True)
%cd /content/ppo-vram-halver

## 2. Import the implementation

The repository module is the source of truth, so this notebook does not keep a separate copy of the training code.

In [ ]:
from rlhf_dual_pass import SingleModelPPOLoss, step_ppo_dual_pass

## 3. Run the GPU memory comparison

The two benchmark arms run in separate processes. The inputs are synthetic; this checks memory behavior, not RLHF training quality. On an 8GB GPU the separate-reference arm may run out of memory.

In [ ]:
!python tests/verify_rlhf.py

## 4. Integrate with a custom PPO loop

See the README for the required rollout tensors, masking rules, and limits. TRL already implements adapter-off reference scoring for PEFT models.

In [ ]:
# Sketch only: collect old_log_probs, old_values, advantages, and returns from a real rollout.
# loss, policy_loss, reference_penalty, value_loss = step_ppo_dual_pass(
#     model=actor_lora_model, inputs=batch_inputs, action_masks=batch_action_masks,
#     old_log_probs=rollout_log_probs, advantages=computed_advantages,
#     returns=computed_returns, optimizer=optimizer,
#     old_values=rollout_values, value_head=critic_head_layer,
# )